# Short Summary Generation

This notebook generates short summaries and geographic scope information from long summaries.
It reads long summary text files and creates structured JSON outputs with:
- Short summary (1-2 sentences)
- Geographic scope (national, regional, municipal, or specific regions/cities)

## 0. Setup

In [ ]:
import json
import time
from pathlib import Path
from openai import OpenAI

# Data directories
DATA_DIR = Path("../data")
SILVER_DIR = DATA_DIR / "silver"
SUMMARIES_DIR = SILVER_DIR / "pad_summaries"
OUTPUT_DIR = SILVER_DIR / "short_summary_json"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Long summaries directory: {SUMMARIES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 1. Load Prompt

In [ ]:
# Load the short summary prompt
prompt_file = Path("../prompts/short_summary.md")
with open(prompt_file, "r", encoding="utf-8") as f:
    short_summary_prompt = f.read()

print(f"Loaded prompt from {prompt_file}")
print(f"Prompt length: {len(short_summary_prompt)} characters")

## 2. Initialize OpenAI Client

In [ ]:
# Initialize OpenAI client
client = OpenAI()

print("✓ OpenAI client initialized")

## 3. Process Long Summaries

In [ ]:
# Get all long summary files
summary_files = list(SUMMARIES_DIR.glob("*_summary.txt"))
print(f"Found {len(summary_files)} long summary files")

# Count existing short summaries
existing_files = list(OUTPUT_DIR.glob("*.json"))
print(f"Existing short summaries: {len(existing_files)}")
print(f"Remaining to process: {len(summary_files) - len(existing_files)}")

In [ ]:
# Process each long summary file
for summary_file in summary_files:
    # Extract project_id from filename
    project_id = summary_file.stem.replace("_summary", "")
    
    # Check if already processed
    output_file = OUTPUT_DIR / f"{project_id}.json"
    if output_file.exists():
        print(f"Skipping {project_id} (already exists)")
        continue
    
    # Read long summary
    with open(summary_file, "r", encoding="utf-8") as f:
        long_summary = f.read()
    
    print(f"Processing {project_id}...")
    
    try:
        # Call OpenAI API with the prompt
        response = client.responses.create(
            prompt={
                "id": "pmpt_695d94ebaef0819796b1eb3d107d00aa053dbcd063f14b6c",
                "version": "2"
            },
            input=long_summary
        )
        
        # Extract the response content
        result_text = response.output_text
        
        # Parse JSON response
        result_json = json.loads(result_text)
        
        # Save to file
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result_json, f, indent=2)
        
        print(f"  ✓ Saved {project_id}")
        
        # Small delay to avoid rate limits
        time.sleep(0.5)
        
    except Exception as e:
        print(f"  ✗ Error processing {project_id}: {e}")
        continue

print("\n✓ Completed processing all projects")

## 4. Verify Output

In [ ]:
# Count final output files
output_files = list(OUTPUT_DIR.glob("*.json"))
print(f"Total short summary JSON files: {len(output_files)}")

# Show sample
if output_files:
    sample_file = output_files[0]
    print(f"\nSample file: {sample_file.name}")
    with open(sample_file, "r", encoding="utf-8") as f:
        sample_data = json.load(f)
    print(json.dumps(sample_data, indent=2))